In [1]:
import numpy as np
import pandas as pd
import scipy.io as sio
import os

import plotly.graph_objects as go; import numpy as np
from plotly_resampler import FigureResampler, FigureWidgetResampler
from plotly.subplots import make_subplots
import itertools

In [2]:
def read_mat_file_as_df(filename):
    print(filename)
    mat = sio.loadmat(filename)

    mat = pd.DataFrame(mat['Data']).T
    # read the meta txt (Struct.txt) file
    meta = filename.replace('Data.mat', 'Struct.txt')
    with open(meta) as f:
        meta = f.readlines()
    # replace , with .
    meta = [x.replace(',', '.') for x in meta]
    # remove newline characters
    meta = [x.strip() for x in meta]
    # remove trailing dots
    meta = [x.rstrip('.') for x in meta]

    # set the column names with length of meta
    mat = mat.iloc[:, :len(meta)]
    mat.columns = meta
    
    return mat

# read all .mat files in data/ and store them in a dictionary
data = {}
for root, dirs, files in os.walk('./../data/raw/Test Data Black Kerdi'):
    for file in files:
        if file.endswith('.mat') and not 'StructData' in file and not 'TicTocData' in file:
            filename = os.path.join(root, file)
            data[filename] = read_mat_file_as_df(filename)

./../data/raw/Test Data Black Kerdi\240923\FullCycle\13-48-23\Data.mat
./../data/raw/Test Data Black Kerdi\240923\FullCycle\14-56-41\Data.mat
./../data/raw/Test Data Black Kerdi\240923\FullCycle\16-51-34\Data.mat


In [3]:
# join the dataframes, speed and ramp
df = []
for key in data.keys():
    if 'FullCycle' in key:
        data[key]['Cycle'] = 'FullCycle'
    data[key]['File'] = key

    df.append(data[key])
df = pd.concat(df)

In [4]:
train_start = pd.to_datetime('2024-09-23 13:48:23')
val_start = pd.to_datetime('2024-09-23 14:56:41')
test_start = pd.to_datetime('2024-09-23 16:51:24')

In [5]:
# add new 'set' column based on the file name (train, validation, test)
df['Set'] = df['File'].apply(lambda id: 'Train' if '13-48-23' in id else 'Validation' if '14-56-41' in id else 'Test' if '16-51-34' in id else 'Unknown')

# group by file and remove the first time stamp of each file
df['Time'] = df['Time'] - df.groupby('File')['Time'].transform('min')

# read Time as datetime
df['Time'] = pd.to_timedelta(df['Time'], unit='ms')

# add train_start, val_start, test_start to the current time column
df['Time'] = df['Time'] + df['Set'].apply(lambda id: train_start if 'Train' in id else val_start if 'Validation' in id else test_start if 'Test' in id else pd.to_datetime('2024-09-23 00:00:00'))

r = 0.074/2
df['WebSpeedReal'] = - df['Encoders.Encoder1.Speed'] * 2*np.pi*r

# add features
#df['Motors.Roller1.WebSpeedSimple'] = df['Motors.Roller1.Speed']*df['AnalogSensors.Radius1Fine']*2*np.pi
df['Slip'] = df['WebSpeedReal'] - df['Motors.Traction1.Speed']

In [6]:
# remove test cycles
df = df[df['Cycle'] != 'TestCycle']
# remove Cycle column
df = df.drop(columns=['Cycle'])
df = df.reset_index(drop=True)

In [7]:
# check nans
df.isnull().sum().sum()

np.int64(0)

### Split by speed levels 

In [8]:
df['RampNumber'] = 0
df['Ramp'] = 0

In [9]:
# determine ramp up and ramp down if positive or negative derivative
for id, g in df.groupby(['File']):
    g['Ramp'] = g['Motors.Traction1.Setpoint'].abs().diff().bfill()
    g['Ramp'] = np.sign(g['Ramp'])
    # pad 
    mask = g['Ramp'] == 0
    g['Ramp'] += g['Ramp'].shift(-1) * mask
    df.iloc[g.index, df.columns.get_loc('Ramp')] = g['Ramp']

# number all positive ramps and negative ramps
mask = df['Ramp'] == 1
df['RampNumber'] = (mask != mask.shift()).cumsum()-1

mask = df['Ramp'] == -1
df['RampNumber'] = df['RampNumber'] + ((mask != mask.shift()).cumsum()-1)

# ramp number 0 where 0
mask = df['Ramp'] == 0
df['RampNumber'] = df['RampNumber'] + ((mask != mask.shift()).cumsum()-1)

# get the largest value of the ramp (positive or negative)
df['RampSpeed'] = 0
df['RampTime'] = 0 # sec
for id, g in df.groupby(['File','Ramp', 'RampNumber']):
    df.loc[g.index, 'RampSpeed'] = max(g['Motors.Traction1.Setpoint'].values, key=abs)
    # if ramp is not 0
    if df.loc[g.index, 'Ramp'].values[0] != 0:
        df.loc[g.index, 'RampTime'] = (g['Time'].max() - g['Time'].min()).total_seconds()

# df['AbsRampSpeed'] = df['RampSpeed'].abs()

# round ramptime to 1 decimal
df['RampTime'] = df['RampTime'].round(0)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_17956\217118652.py:28: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '10.02' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[g.index, 'RampTime'] = (g['Time'].max() - g['Time'].min()).total_seconds()


In [10]:
# Create a new column initialized to NaN
df['PulseNumber'] = 0
pulse_number = 1
i = 0
while i < len(df) - 2:
    if df['Ramp'].iloc[i] == 1:
        start = i
        # Block of 1s
        while i < len(df) and df['Ramp'].iloc[i] == 1:
            i += 1
        # Block of 0s
        if i < len(df) and df['Ramp'].iloc[i] == 0:
            while i < len(df) and df['Ramp'].iloc[i] == 0:
                i += 1
            # Block of -1s
            if i < len(df) and df['Ramp'].iloc[i] == -1:
                while i < len(df) and df['Ramp'].iloc[i] == -1:
                    i += 1
                # Full pattern found: assign PulseNumber
                df.loc[start:i, 'PulseNumber'] = pulse_number
                pulse_number += 1
                continue
    i += 1

In [11]:
# Groupby pulsenumber and add 0.5 second before and after the pulse
for id, g in df.groupby('PulseNumber'):
    # Add 500 ms before and after the pulse
    start_time = g['Time'].min() - pd.Timedelta(seconds=1)
    end_time = g['Time'].max() + pd.Timedelta(seconds=1)
    # Assign the PulseNumber into the dataframe
    df.loc[(df['Time'] >= start_time) & (df['Time'] <= end_time), 'PulseNumber'] = id
    # Cut out the middle part
    start_time = g[g['Ramp'] == 1]['Time'].max() + pd.Timedelta(seconds=2.5)
    end_time = g[g['Ramp'] == -1]['Time'].min() - pd.Timedelta(seconds=2.5)
    df.loc[(df['Time'] >= start_time) & (df['Time'] <= end_time), 'PulseNumber'] = 0

In [12]:
df['PulseSpeed'] = 0
df['PulseRampTime'] = 0
df['PulseDirection'] = 0
for id, g in df.groupby('PulseNumber'):
    if id == 0:
        continue
    df.loc[g.index, 'PulseSpeed'] = g['RampSpeed'].abs().max()
    df.loc[g.index, 'PulseDirection'] = np.sign(max(g['RampSpeed'].values, key=abs))
    df.loc[g.index, 'PulseRampTime'] = g['RampTime'].max()

In [13]:
ramp_groups = {}
for (pulse_set, speed, ramp_time, direction, pulse_number), g in df.groupby(['Set', 'PulseSpeed', 'PulseRampTime', 'PulseDirection', 'PulseNumber']):
    if speed == 0 or ramp_time == 0 or pulse_number == 0 or speed == 10 or ramp_time == 10:
        continue
    if pulse_number == 221:
        pulse_df = df.loc[df['PulseNumber'] == 223]
        pulse_df['PulseNumber'] = pulse_number
        pulse_df['Time'] = pd.date_range(start=g['Time'].min(), periods=pulse_df.shape[0], freq=f'{20}L')
        ramp_groups[(pulse_set, speed, ramp_time, direction, pulse_number)] = pulse_df
    elif pulse_number == 222:
        pulse_df = df.loc[df['PulseNumber'] == 224]
        pulse_df['PulseNumber'] = pulse_number
        pulse_df['Time'] = pd.date_range(start=g['Time'].min(), periods=pulse_df.shape[0], freq=f'{20}L')
        ramp_groups[(pulse_set, speed, ramp_time, -1, pulse_number)] = pulse_df
    else:
        ramp_groups[(pulse_set, speed, ramp_time, direction, pulse_number)] = g

C:\Users\Administrator\AppData\Local\Temp\ipykernel_17956\215532667.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pulse_df['PulseNumber'] = pulse_number
C:\Users\Administrator\AppData\Local\Temp\ipykernel_17956\215532667.py:8: FutureWarning: 'L' is deprecated and will be removed in a future version, please use 'ms' instead.
  pulse_df['Time'] = pd.date_range(start=g['Time'].min(), periods=pulse_df.shape[0], freq=f'{20}L')
C:\Users\Administrator\AppData\Local\Temp\ipykernel_17956\215532667.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexin

In [14]:
# Extract unique first elements from tuple keys
set_list = set([key[0] for key in ramp_groups.keys()])
set_dict = {unique_set: [] for unique_set in set_list}
for unique_set in set_list:
    set_dict[unique_set] = pd.concat([g for key, g  in ramp_groups.items() if key[0] == unique_set]).sort_values(by='Time').reset_index(drop=True)

In [15]:
gap_list = [(g.index[0], g.index[-1]) for (speed, ramp_time), g in set_dict['Test'].groupby(['PulseSpeed', 'PulseRampTime']) if speed == 60]
gap_dict = {ramp_time[0]:g.shape[0] for ramp_time, g in set_dict['Test'].groupby(['PulseRampTime'])}

In [16]:
def shift_dataframe_zeros(df: pd.DataFrame, gap_list: list) -> pd.DataFrame:
    """
    """
    # Sort gap_list in ascending order of begin_index.
    sorted_gaps = sorted(gap_list, key=lambda x: x[0])
    padded_df = df.copy()

    for original_begin, original_end in sorted_gaps[:-1]:
        num_zeros = original_end - original_begin
        # First add rows equal to the gap
        empty_rows = pd.DataFrame(columns=df.columns, index=range(num_zeros))
        # Concatenate with the original DataFrame
        padded_df = pd.concat([padded_df, empty_rows], ignore_index=True)
        # TODO: FIX Then shift
        for col in padded_df.columns:
            padded_df.iloc[original_begin + num_zeros:][col].values[:] = padded_df.iloc[original_begin:-num_zeros][col].values[:]
        # Then fill in zeros
        for col in padded_df.columns:
            padded_df.iloc[original_begin:original_end][col].values[:] = 0
    return padded_df

In [17]:
def pad_dataframe_zeros(df: pd.DataFrame, num_zeros: dict) -> pd.DataFrame:
    """
    """
    # First add rows equal to the gap
    zero_rows = pd.DataFrame(0, index=range(num_zeros), columns=df.columns)
    # Concatenate with the original DataFrame
    padded_df = pd.concat([df, zero_rows], ignore_index=True)
    return padded_df

In [18]:
row_order_dict = {'Train': 1, 'Validation': 2, 'Test': 3}
col_order_dict = {8:1, 6:2, 4:3, 2:4}
subplot_list = [f'{r[0]}_{r[1]}s' for r in itertools.product(row_order_dict.keys(), col_order_dict.keys())]
fig = FigureResampler(make_subplots(rows=3, cols=4, subplot_titles=subplot_list, shared_xaxes='all', shared_yaxes='all'))
for unique_set, set_g in set_dict.items():
    for ramp_time, g in set_g.groupby('PulseRampTime'):
        if unique_set != 'Test':
            g = pad_dataframe_zeros(g, gap_dict[ramp_time]-g.shape[0])
        else:
            g = g.reset_index()

        fig.add_trace(go.Scatter(name=unique_set + '_WebSpeedReal', line_color='blue'), hf_x=g.index, hf_y=g['WebSpeedReal'], row=row_order_dict[unique_set], col=col_order_dict[ramp_time])
        # fig.add_trace(go.Scatter(name=id + '_Motors.Traction1.Speed'), hf_x=g.index, hf_y=g['Motors.Traction1.Speed'], row=order_dict[id], col=1)
        # fig.add_trace(go.Scatter(name=id + '_Motors.Traction1.Setpoint'), hf_x=g.index, hf_y=g['Motors.Traction1.Setpoint'], row=order_dict[id], col=1)
        fig.add_trace(go.Scatter(name=unique_set + '_Slip', line_color='red'),hf_x=g.index, hf_y=g['Slip'], row=row_order_dict[unique_set], col=col_order_dict[ramp_time])
        # Add vertical lines between each speed
        for (speed), cur_df in g.groupby('PulseSpeed'):
            fig.add_vline(x=cur_df.index[-1], line_width=1, line_dash='dash', line_color='black', row=row_order_dict[unique_set], col=col_order_dict[ramp_time])

# set title
for row in range(1, 4):
    for col in range(1, 5):
        fig.update_yaxes(range=[-65, 65], tickvals=[-60, -50, -40, -30, -20, 0, 20, 30, 40, 50, 60], row=row, col=col)

for col in range(1, 5):
    fig.update_xaxes(title='Time', row=3, col=col)

for row in range(1, 4):
    fig.update_yaxes(title='Speed (m/min)', row=row, col=1)

fig.update_layout(height=800, template='plotly_white')
# Remove legend
fig.update_layout(showlegend=False)
fig.show()
# Save figure as pdf
fig.write_image(format="pdf", file="./../figures/ramp_pulses_speeds.pdf", height=800, width=1100)

In [19]:
# Save files per set
for unique_set, g in set_dict.items():
    g.to_parquet(f'./../data/processed/processed_{unique_set}.parquet'.lower(), index=False)